# Ecopath with Ecosim from Python: a Colab quickstart

This notebook runs the EwE engine from Python with the `ewe` package. It walks through:

1. Installing the `ewe` wheel (the engine, .NET and the model library are bundled, nothing else to install)
2. Loading a model from the built-in library
3. Running Ecopath (mass balance) and Ecosim (time dynamics), and plotting the results
4. Changing fishing effort and a model parameter, and rerunning
5. Saving the edited model (`.eiixml`, or an Access `.ewemdb` that opens in the EwE desktop app)
6. Equilibrium yield curves (MSY / FMSY)
7. Fitting Ecosim to time series

It runs in Google Colab (Linux x86-64) and in any local Jupyter with the wheel for your platform.
Every run takes a few seconds on the small *Anchovy Bay* teaching model.

## 1. Install

`WHEEL_URL` points at the Linux x86-64 wheel (about 70 MB); the `[plot]` extra adds matplotlib for the charts. On your own machine, install the wheel for
your platform instead (`pip install ewe-0.1.0-py3-none-<platform>.whl`) and skip this cell.
If the URL is empty, the cell asks you to upload the `.whl` file instead.

In [ ]:
WHEEL_URL = "https://github.com/dexterfichuk/ewe-wheels/releases/download/v0.1.0/ewe-0.1.0-py3-none-manylinux_2_28_x86_64.whl"

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("ewe") is None:
    if WHEEL_URL:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ewe[plot] @ {WHEEL_URL}"], check=True)
    else:
        from google.colab import files  # upload the .whl from your computer
        wheel = next(iter(files.upload()))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ewe[plot] @ file://{os.path.abspath(wheel)}"], check=True)

In [ ]:
import ewe
import ewe.library
import pandas as pd

pd.set_option("display.precision", 3)
print("ewe", ewe.__version__)

## 2. Load a library model

The wheel ships a small library of published and teaching models. `ewe.library.load` opens one by id;
`ewe.open("my_model.eiixml")` opens your own file (upload it in the Colab file pane first).

In [ ]:
ewe.library.list()

In [ ]:
m = ewe.library.load("anchovy-bay")
print(m)
print(m.engine_version)
m.fleets

## 3. Ecopath: mass balance

`ecopath()` balances the model and returns one row per group. `diagnostics` holds the columns you
check when balancing: trophic level, biomass, P/B, Q/B, ecotrophic efficiency (EE must be at most 1),
gross efficiency, fishing and predation mortality.

Every result has a `plot` accessor that draws the usual chart and returns the matplotlib Axes, so you
can restyle or save it (`ax.figure.savefig("chart.png")`).

In [ ]:
path = m.ecopath()
print("balanced:", path.balanced, "| unbalanced groups:", path.unbalanced_groups)
path.diagnostics

In [ ]:
path.plot();   # biomass against trophic level

In [ ]:
m.plot.foodweb();   # flow diagram: trophic level up, node size = biomass, links = consumption

The diet matrix and catches the balance used are available too:

In [ ]:
inputs = m.ecopath_inputs()
inputs.diet_matrix.round(3)

## 4. Ecosim: time dynamics

`ecosim()` runs the model's first Ecosim scenario with its stored effort and forcing. Results are long
tables (`groups`, `fleets`) with monthly steps; `biomass`, `catch` and `relative_biomass` pivot them to
time x group. Producers and their grazers cycle within each year, so the plots average to years with
averaging each year (`annual=True`, the default).

In [ ]:
sim = m.ecosim()
print(sim.scenario, "|", sim.years, "years")
sim.plot.biomass(relative=True);

In [ ]:
sim.plot.fleets();   # landings by fleet, stacked

### What if the seiners fished twice as hard?

`effort` multiplies a fleet's whole effort series for this run only; the model is not changed.

In [ ]:
doubled = m.ecosim(effort={"Seiners": 2})
sim.plot.compare({"seiner effort x2": doubled}, label="baseline", groups=["Anchovy", "Mackerel", "Cod", "Whiting"]);

(doubled.biomass.iloc[-1] / sim.biomass.iloc[-1]).rename("end biomass, x2 seiners / baseline").to_frame().T

## 5. Edit a parameter and rerun

`m.document()` returns the model as an editable document (groups, diet, fleets, catch, stanzas,
scenarios). `m.apply(doc)` makes the open session use the edit; the file on disk is untouched until
you save. Here we lower the anchovy's production rate (P/B) from 1.15 to 0.9 per year, which raises
the share of its production that predators and fishing use (EE).

In [ ]:
doc = m.document()
doc.groups.set_index("name").loc[["Anchovy", "Mackerel", "Cod"], ["biomass_area", "pb", "qb", "ee"]]

In [ ]:
doc.set_group("Anchovy", pb=0.9)
m.apply(doc)

edited = m.ecopath()
print("balanced:", edited.balanced, "| unbalanced groups:", edited.unbalanced_groups)
pd.DataFrame({"EE before": path.diagnostics.ee, "EE after": edited.diagnostics.ee}).loc[["Anchovy", "Cod", "Whiting", "Mackerel"]]

Predators and the fishery now take more anchovy than it produces (EE > 1), so the model no longer
balances. Predation in Ecopath depends on the predators' consumption, not on the prey's biomass, so one
fix is a higher anchovy biomass: at 9.5 t/km² its production covers the demand again.
`doc.validate()` reports input errors (missing values, diets that don't sum to 1) before you apply anything.

In [ ]:
doc.set_group("Anchovy", biomass_area=9.5)
m.apply(doc)
edited = m.ecopath()
print("balanced:", edited.balanced)
edited.diagnostics.loc[["Anchovy"], ["biomass", "pb", "ee", "f", "m2"]]

In [ ]:
sim_edit = m.ecosim()
sim_edit.plot.compare({"original": sim}, label="edited (P/B 0.9, B 9.5)", groups=["Anchovy", "Mackerel", "Cod", "Whiting"]);

## 6. Save

`save` writes `.eiixml` (EwE's XML format), or an EwE Access database when the name ends in
`.ewemdb`; that file opens in the EwE desktop app on Windows. No Access driver or Windows is needed.
In Colab, download the files from the file pane or with `files.download`.

In [ ]:
saved = m.save("anchovy_bay_edited.eiixml")
access = m.save("anchovy_bay_edited.ewemdb")
print(saved, saved.stat().st_size // 1024, "kB")
print(access, access.stat().st_size // 1024, "kB")

# Reopen the saved file to check the edit is in it
with ewe.open(saved) as check:
    print("Anchovy in the saved model:", check.ecopath().diagnostics.loc["Anchovy", ["biomass", "pb", "ee"]].to_dict())

try:
    from google.colab import files
    # files.download(str(access))   # uncomment to download
except ImportError:
    pass

## 7. Maximum sustainable yield

`msy` runs EwE's MSY tool: Ecosim is run to equilibrium at a range of fishing mortalities for one group,
under full compensation (other groups' F stays as is) and a stationary system. `max_rel_f` is the
highest F as a multiple of the group's current F.

In [ ]:
m = ewe.library.load("anchovy-bay")   # back to the original model
msy = m.msy("Anchovy", max_rel_f=10)
msy.plot();   # catch and biomass against F; base F and the optimum of each assessment marked
msy.optimum

`fmsy` does the same for every fished group, one at a time:

In [ ]:
fm = m.fmsy()
fm.plot();   # base F against FMSY for each fished group
g = fm.groups
g[g.assessment == "full_compensation"][["group", "f_base", "fmsy", "found", "catch_base", "msy"]]

## 8. Fit Ecosim to time series

The model carries time series datasets: observed relative biomass, catches and fishing effort. `fit`
runs EwE's *Fit to time series* search over predator-prey vulnerabilities and reports the sum of
squares (SS) before and after. Nothing is stored until you call `apply_fit` (this session) or
`save_fit` (a file).

Like the desktop app, `fit` starts from the default vulnerability (2) on every link. Anchovy Bay already
carries tuned vulnerabilities, so here the search starts from those instead (`reset_vulnerabilities=False`).
Add `anomaly_spline_points=5` to also search a primary production anomaly.

In [ ]:
ts = m.time_series(dataset="anchovybay")
print(m.time_series().datasets)
ts.series.groupby(["series", "type", "target"]).size().rename("points").to_frame()

In [ ]:
fit = m.fit(dataset="anchovybay", reset_vulnerabilities=False)
print(f"SS before {fit.ss_before:.3f}, after {fit.ss_after:.3f}, AIC {fit.aic:.1f}, "
      f"{fit.n_parameters} parameters, {len(fit.iterations)} iterations")
fit.plot();   # sum of squares by iteration
fit.changed.sort_values("after", ascending=False).head(10)

Compare the observations with Ecosim before and after the fit. The dataset's first year lines up with
the first year of the scenario, and relative indices are scaled to the prediction the way EwE scores
them (the geometric mean of predicted / observed).

In [ ]:
before = m.ecosim()
m.apply_fit(fit)
after = m.ecosim()
ts.plot.fit({"before fit": before, "after fit": after});

In [ ]:
fitted = m.save_fit(fit, "anchovy_bay_fitted.eiixml")
print("saved", fitted)
m.close()

## More

The same `Model` also runs Monte Carlo (`m.monte_carlo`), Ecospace (`m.ecospace`), Ecotracer,
management strategy evaluation (`m.mse`), policy search (`m.policy_search`), network analysis
(`m.network_analysis`) and ecological indicators (`m.indicators`). See `python/README.md` in the
repository for each call, and `ewe.new_model(...)` to build a model from scratch.